# Official evaluation of existing predictions

Use the unchanged OpenProblems metric-components and metadata exporter. No metric formulas are implemented here.

The default comparison is **BABEL 20 epochs vs existing official mean-per-gene scores** on the development/validation partition. No training runs; the final test partition is not used.

Run the input check first. Stop if identifiers or order differ. The first real server evaluation is still pending.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

repo = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src/workflows/run_benchmark/config.vsh.yaml").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the benchmark repository.")
sys.path.insert(0, str(repo / "scripts"))
import official_evaluation as evaluation
print("Repository:", repo)


## 1. Settings
`predictions` can contain any number of methods/datasets using the benchmark metric interface. Metrics are discovered from the workflow's `metrics/` dependencies, not from a fixed list in this notebook.

The image overrides below reproduce the working baseline environment; add an override for a new component only if its generated image is unavailable. For a fresh attempt, change `output_dir`.

`scale_confirmed` must reflect the actual prediction code/run settings. Merely having a layer named `normalized` does not prove that its values are on the target scale. Leave this false until the first input review is complete.


In [ ]:
settings = {
    "build_dir": "target_benchmark_complete",
    "output_dir": "outputs/tchic/evaluations/h3k4me3_genelevel_dev/official_001",
    "nextflow_version": "26.04.4",  # version in the successful baseline notebook
    "container_overrides": {
        "correlation": "ghcr.io/openproblems-bio/task_predict_modality/metrics/correlation:build_main",
        "mse": "ghcr.io/openproblems-bio/task_predict_modality/metrics/mse:build_main",
    },
    "predictions": [{
        "run": "babel_20epochs",
        "prediction": "outputs/tchic/runs/babel_20epochs/babel_prediction.h5ad",
        "target": "outputs/tchic/prepared/h3k4me3_genelevel_dev/test_mod2.h5ad",
        "normalization": "log_cp10k",
        "scale_confirmed": True,
        "scale_evidence": "Reviewed babel_predict/script.py and generated main.nf: per-cell scaling to 1e4, then log1p, stored in layers['normalized'].",
    }],
    "existing_scores": [{
        "run": "mean_per_gene_existing",
        "path": "outputs/tchic/runs/h3k4me3_genelevel_dev/mean_per_gene/publish/score_uns.yaml",
        "dataset_id": "tchic_mgast_h3k4me3_genelevel_dev",
        "method_id": "mean_per_gene",
    }],
}


## 2. Check inputs — no evaluation is started
Checks shapes, exact cell/gene IDs and order, dataset identity and declared normalization. Stops on mismatch; does not modify or reorder files.

**For the first use, share the output of this cell before proceeding.** If prediction normalization metadata is absent, verify its scale from the prediction code/run settings before setting `scale_confirmed=True`.


In [ ]:
metrics = evaluation.discover_metrics(repo, repo / settings["build_dir"])
display(pd.DataFrame([{"component": m["name"], "metrics": ", ".join(m["metric_ids"])} for m in metrics]))
checks = []
for job in settings["predictions"]:
    checks.append({"run": job["run"], **evaluation.inspect_pair(
        repo / job["target"], repo / job["prediction"], job["normalization"]
    )})
display(pd.DataFrame(checks))
for item in settings["existing_scores"]:
    baseline = evaluation.score_rows(evaluation.read_yaml(repo / item["path"]),
        item["run"], item["dataset_id"], item["method_id"])
    display(pd.DataFrame(baseline))


## 3. Save an execution plan
After reviewing the inputs and confirming the prediction scale, this records source/build/input hashes, software version, settings, runtime configuration and exact commands. It does not run metrics. The output directory must be new.

The same saved plan is executable outside Jupyter through its `run_command.sh`. Container tags remain mutable: retain the Singularity/Apptainer image cache for exact replay. Full container locking is not yet implemented.


In [ ]:
plan = evaluation.prepare(repo, settings)
display(pd.DataFrame([{k: s[k] for k in ("run", "metric", "kind", "output")} for s in plan["steps"]]))
print("Saved run command:", Path(plan["output"]) / "run_command.sh")


## 4. Execute official components
This runs metrics and the existing metadata exporter for each prediction. Every step has a separate `console.log`, Nextflow log, work directory and trace. A failed/missing/partial result stops the run. A plan is single-use; choose a new output directory after failure.

The baseline scores are read from their existing official YAML, not recalculated. To re-evaluate a baseline prediction later, add its verified prediction/target pair to `predictions` just like any other method.


In [ ]:
scores = evaluation.execute(plan)
print("Completed:", plan["output"])


## 5. Read saved results
This cell also works after restarting the kernel (run imports/settings first). Each new prediction run has a `score_uns.yaml` using the original exported `.uns` records. CSV files are analysis views; they do not replace the official component outputs.


In [ ]:
import json
output = repo / settings["output_dir"]
status = json.loads((output / "status.json").read_text())
if status["state"] != "complete":
    raise RuntimeError(f"Evaluation is not complete: {status}")
scores = pd.read_csv(output / "scores_long.csv")
display(scores.pivot(index=["run", "method_id"], columns="metric", values="value"))
display(scores[["run", "evaluation", "score_source"]].drop_duplicates())


## 6. Standard development plots
Each metric has its own axis. These are validation scores, not final-test performance. Genewise correlations for constant predictions follow the upstream zero-score convention.


In [ ]:
import matplotlib.pyplot as plt
selected = ["rmse", "mean_pearson_per_cell", "mean_spearman_per_cell"]
fig, axes = plt.subplots(1, len(selected), figsize=(12, 3.6))
for axis, metric in zip(axes, selected):
    subset = scores.loc[scores["metric"] == metric]
    axis.barh(subset["run"], subset["value"], color="#0072B2")
    axis.set_title(metric.replace("_", " "))
    axis.set_xlabel("lower is better" if metric == "rmse" else "higher is better")
    axis.grid(axis="x", alpha=0.25)
fig.tight_layout()
fig.savefig(output / "development_scores.png", dpi=160, bbox_inches="tight")
plt.show()
